Mount Google Drive and initialize project

In [ ]:
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Project configuration
PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

# Set working directory
os.chdir(PROJECT_DIR)

print("Project mounted successfully!")
print("Working directory:", os.getcwd())
print("Python version:", sys.version)

Mounted at /content/drive
Project mounted successfully!
Working directory: /content/drive/MyDrive/DumLLM
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


Install uv

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
    check=True
)

print("uv installed successfully!")

uv installed successfully!


Install all dependencies

In [ ]:
import subprocess

subprocess.run(
    [
        "uv", "pip", "install",
        "--system",
        "-r", "requirements.txt"
    ],
    check=True
)

print("All dependencies installed successfully!")

All dependencies installed successfully!


Verify environment

In [ ]:
import sys
import os
import subprocess

print("Python:", sys.version)
print("Executable:", sys.executable)

subprocess.run(["uv", "--version"], check=True)
subprocess.run(["uv", "pip", "list"], check=True)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Executable: /usr/bin/python3


CompletedProcess(args=['uv', 'pip', 'list'], returncode=0)

Verify important libraries

In [ ]:
import torch
import numpy as np
import pandas as pd

print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nDumLLM environment is ready!")

PyTorch: 2.11.0+cu130
NumPy: 2.1.3
Pandas: 2.2.3
CUDA available: True
GPU: Tesla T4

DumLLM environment is ready!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/DumLLM")
SRC_DIR = PROJECT_DIR / "src"

# Create src directory if missing
SRC_DIR.mkdir(parents=True, exist_ok=True)

# Create setup utility
setup_code = '''
import os

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

def setup_drive():
    os.chdir(PROJECT_DIR)
    print("DumLLM initialized!")
    print("Working directory:", os.getcwd())
'''

(SRC_DIR / "colab_setup.py").write_text(setup_code)

print("Created:", SRC_DIR / "colab_setup.py")

Mounted at /content/drive
Created: /content/drive/MyDrive/DumLLM/src/colab_setup.py


In [ ]:
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

subprocess.run(
    ["git", "config", "user.name", "Shubham Kumar Sahu"],
    cwd=PROJECT_DIR,
    check=True
)

subprocess.run(
    ["git", "config", "user.email", "shubhamisok@gmail.com"],
    cwd=PROJECT_DIR,
    check=True
)

print("Git identity configured!")

Git identity configured!


In [ ]:
from google.colab import userdata
import subprocess
import base64

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

token = userdata.get("GITHUB_TOKEN")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

result = subprocess.run(
    [
        "git",
        "-c", f"http.extraheader=AUTHORIZATION: basic {auth}",
        "push", "-u", "origin", "main"
    ],
    cwd=PROJECT_DIR
)

if result.returncode == 0:
    print("Successfully pushed to GitHub!")
else:
    print("Push failed. Check your token permissions.")

Successfully pushed to GitHub!


#GitHub Auto Sync

In [ ]:
import os
import sys
import subprocess
import base64
from getpass import getpass
from google.colab import userdata

# ==========================================
# DumLLM - GitHub Auto Sync
# ==========================================

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"
REPO_URL = "https://github.com/ShubhamKumarSahu-svg/DumLLM.git"
BRANCH = "main"

os.chdir(PROJECT_DIR)

def run(command, check=True):
    return subprocess.run(
        command,
        check=check,
        text=True,
        capture_output=True
    )

# 1. Verify Git repository
if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    raise RuntimeError("Git repository not found!")

# 2. Configure remote
remotes = run(["git", "remote"]).stdout.split()

if "origin" not in remotes:
    run(["git", "remote", "add", "origin", REPO_URL])
else:
    run(["git", "remote", "set-url", "origin", REPO_URL])

# 3. Protect unnecessary files
gitignore = os.path.join(PROJECT_DIR, ".gitignore")

ignore_rules = [
    ".venv/",
    "venv/",
    "__pycache__/",
    "*.py[cod]",
    ".env",
    ".env.*",
    "!.env.example",
    ".ipynb_checkpoints/",
]

existing = ""
if os.path.exists(gitignore):
    with open(gitignore, "r") as f:
        existing = f.read()

with open(gitignore, "a") as f:
    for rule in ignore_rules:
        if rule not in existing.splitlines():
            f.write(f"\n{rule}")

# Remove virtual environment from Git tracking if previously added
tracked = run(
    ["git", "ls-files", "--", ".venv", "venv"],
).stdout.strip()

if tracked:
    run(["git", "rm", "-r", "--cached", "--", ".venv", "venv"])

# 4. Get commit message
commit_message = input("Enter commit message: ").strip()

if not commit_message:
    raise ValueError("Commit message cannot be empty!")

# 5. Stage all changes
run(["git", "add", "-A"])

# 6. Check for changes
status = run(["git", "diff", "--cached", "--quiet"], check=False)

if status.returncode == 0:
    print("No changes to commit. Everything is up to date!")
else:
    # 7. Commit
    run(["git", "commit", "-m", commit_message])
    print("Commit created successfully!")

    # 8. Get token from Colab Secrets
    token = userdata.get("GITHUB_TOKEN")

    if not token:
        raise RuntimeError("GITHUB_TOKEN not found in Colab Secrets!")

    auth = base64.b64encode(
        f"x-access-token:{token}".encode()
    ).decode()

    # 9. Push securely without saving credentials
    result = subprocess.run(
        [
            "git",
            "-c", f"http.extraheader=AUTHORIZATION: basic {auth}",
            "push", "-u", "origin", f"HEAD:{BRANCH}"
        ],
        cwd=PROJECT_DIR,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError("GitHub push failed. Check authentication and branch.")

    print("\nSuccessfully pushed to GitHub!")
    print(f"Repository: {REPO_URL}")
    print(f"Branch: {BRANCH}")
    print(f"Commit: {commit_message}")

print("\nDumLLM GitHub sync complete!")